# InternVL2.5-2B Colab baseline (LoRA + next-token MC scoring)

이 노트북은 SSAFY AI 챌린지 규칙 범위 안에서 다음만 사용합니다.

- Hugging Face 사전학습 모델: `OpenGVLab/InternVL2_5-2B`
- 로컬/Colab 내 파인튜닝 및 추론
- 외부 API 호출 없음
- 제출용 `submission.csv` 생성

실행 전 준비:
- `train.csv`, `test.csv`, `train/`, `test/`를 `/content` 아래에 두기
- 또는 코드 상단의 `DATA_ROOT_CANDIDATES`에 맞게 경로 수정


In [ ]:
# 1) Run this cell first, then Runtime -> Restart session once.
!pip uninstall -y transformers tokenizers accelerate peft bitsandbytes pillow -q
!pip install -q "torch==2.5.1" "torchvision==0.20.1" --index-url https://download.pytorch.org/whl/cu121
!pip install -q "transformers==4.48.3" "accelerate==1.2.1" "peft==0.14.0" "bitsandbytes==0.45.2"     "pillow==10.4.0" "pandas==2.2.3" "numpy==2.0.2" "scikit-learn==1.6.1" "tqdm==4.67.1"     "einops>=0.8.0" "timm>=1.0.12" "sentencepiece>=0.2.0"


In [ ]:

# Colab-ready InternVL2.5-2B LoRA training / inference for SSAFY VQA multiple-choice challenge
# - Follows competition rules: no API inference, uses Hugging Face pretrained model + local fine-tuning
# - Fails fast with explicit errors instead of silent fallbacks
#
# Recommended runtime:
#   - Colab GPU (T4 / L4 / A100)
#   - Python 3.10+
#
# Install FIRST in a separate Colab cell:
# !pip uninstall -y transformers tokenizers accelerate peft bitsandbytes pillow
# !pip install -q "torch==2.5.1" "torchvision==0.20.1" --index-url https://download.pytorch.org/whl/cu121
# !pip install -q "transformers==4.48.3" "accelerate==1.2.1" "peft==0.14.0" "bitsandbytes==0.45.2" \
#     "pillow==10.4.0" "pandas==2.2.3" "numpy==2.0.2" "scikit-learn==1.6.1" "tqdm==4.67.1" \
#     "einops>=0.8.0" "timm>=1.0.12" "sentencepiece>=0.2.0"

import os
import gc
import math
import copy
import json
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
from PIL import Image, ImageFile
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import InterpolationMode
from tqdm.auto import tqdm

from transformers import (
    AutoModel,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


# =========================
# 0. Hard checks / config
# =========================
SEED = 42
MODEL_ID = "OpenGVLab/InternVL2_5-2B"

# Data root candidates checked in order.
DATA_ROOT_CANDIDATES = [
    "/content",
    "/content/data",
    "/content/drive/MyDrive",
    "/content/drive/MyDrive/ssafy_ai",
]

TRAIN_CSV_NAME = "train.csv"
TEST_CSV_NAME = "test.csv"

OUTPUT_DIR = "/content/internvl2_5_2b_lora_run"

IMAGE_SIZE = 448
MAX_PATCHES = 6             # official 2nd-finetune shell for InternVL2.5-2B uses max_dynamic_patch=6
USE_THUMBNAIL = True
NUM_EPOCHS = 2
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
LR = 4e-5                  # official 2nd-finetune shell uses 4e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0
NUM_WORKERS = 2
VAL_SIZE = 0.1
LORA_R = 16                # official shell uses --use_llm_lora 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
USE_FLASH_ATTN = False     # keep False for Colab stability; do not install flash-attn implicitly
MAX_SAMPLES = None         # set integer for quick debugging, e.g. 256
SAVE_SUBMISSION_NAME = "submission.csv"

SYSTEM_MESSAGE = (
    "You are a visual multiple-choice assistant. "
    "You must answer with exactly one lowercase letter among a, b, c, d. "
    "Do not output any other words, punctuation, or explanation."
)

PROMPT_SUFFIX = (
    "보기:\n"
    "(a) {a}\n"
    "(b) {b}\n"
    "(c) {c}\n"
    "(d) {d}\n\n"
    "정답은 반드시 소문자 한 글자(a, b, c, d)로만 답하세요."
)

Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = False


def fail(msg: str) -> None:
    raise RuntimeError(msg)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def detect_dtype() -> torch.dtype:
    if not torch.cuda.is_available():
        fail("CUDA GPU가 필요합니다. Colab 런타임을 GPU로 변경한 뒤 다시 실행하세요.")
    major, _minor = torch.cuda.get_device_capability()
    # T4 = sm75 -> bf16 미지원
    return torch.bfloat16 if major >= 8 else torch.float16


def resolve_data_root() -> Path:
    checked = []
    for root in DATA_ROOT_CANDIDATES:
        root = Path(root)
        checked.append(str(root))
        if (root / TRAIN_CSV_NAME).exists() and (root / TEST_CSV_NAME).exists():
            return root
    fail(
        "train.csv / test.csv 를 찾지 못했습니다.\n"
        f"확인한 경로:\n- " + "\n- ".join(checked)
    )


def validate_dataframe(df: pd.DataFrame, is_train: bool) -> None:
    required = ["id", "path", "question", "a", "b", "c", "d"]
    if is_train:
        required.append("answer")
    missing = [c for c in required if c not in df.columns]
    if missing:
        fail(f"CSV 컬럼이 부족합니다. 누락 컬럼: {missing}")
    if is_train:
        bad = df[~df["answer"].astype(str).str.lower().isin(list("abcd"))]
        if len(bad) > 0:
            fail(f"train.csv answer 컬럼에 a/b/c/d 외 값이 있습니다. 예시: {bad[['id','answer']].head(5).to_dict('records')}")


# =========================
# 1. Official InternVL image preprocessing
#    (adapted from official quick start)
# =========================
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def build_transform(input_size: int) -> T.Compose:
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def find_closest_aspect_ratio(
    aspect_ratio: float,
    target_ratios: List[Tuple[int, int]],
    width: int,
    height: int,
    image_size: int,
) -> Tuple[int, int]:
    best_ratio_diff = float("inf")
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio


def dynamic_preprocess(
    image: Image.Image,
    min_num: int = 1,
    max_num: int = MAX_PATCHES,
    image_size: int = IMAGE_SIZE,
    use_thumbnail: bool = USE_THUMBNAIL,
) -> List[Image.Image]:
    orig_width, orig_height = image.size
    if orig_width <= 0 or orig_height <= 0:
        fail(f"잘못된 이미지 크기입니다: width={orig_width}, height={orig_height}")

    aspect_ratio = orig_width / orig_height
    target_ratios = set(
        (i, j)
        for n in range(min_num, max_num + 1)
        for i in range(1, n + 1)
        for j in range(1, n + 1)
        if i * j <= max_num and i * j >= min_num
    )
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size
    )
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    resized_img = image.resize((target_width, target_height))
    processed_images = []
    cols = target_width // image_size
    rows = target_height // image_size

    for i in range(blocks):
        box = (
            (i % cols) * image_size,
            (i // cols) * image_size,
            ((i % cols) + 1) * image_size,
            ((i // cols) + 1) * image_size,
        )
        processed_images.append(resized_img.crop(box))

    if len(processed_images) != blocks:
        fail(f"패치 수 불일치: expected={blocks}, got={len(processed_images)}")

    if use_thumbnail and len(processed_images) != 1:
        processed_images.append(image.resize((image_size, image_size)))

    return processed_images


def load_image_tensor(image_path: Path, input_size: int = IMAGE_SIZE, max_num: int = MAX_PATCHES) -> torch.Tensor:
    if not image_path.exists():
        fail(f"이미지 파일을 찾지 못했습니다: {image_path}")
    image = Image.open(image_path).convert("RGB")
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=USE_THUMBNAIL, max_num=max_num)
    pixel_values = [transform(tile) for tile in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values


# =========================
# 2. Prompt / template utils
# =========================
def build_user_question(row: pd.Series) -> str:
    return (
        f"{str(row['question']).strip()}\n\n" +
        PROMPT_SUFFIX.format(
            a=str(row["a"]).strip(),
            b=str(row["b"]).strip(),
            c=str(row["c"]).strip(),
            d=str(row["d"]).strip(),
        )
    )


def get_conv_template_instance(model: Any) -> Any:
    if not hasattr(model, "conv_template"):
        fail("model.conv_template 을 찾지 못했습니다. InternVL custom code 로드에 실패했을 수 있습니다.")
    return copy.deepcopy(model.conv_template)


def build_conversation_text(
    model: Any,
    user_question: str,
    answer: Optional[str] = None,
) -> str:
    template = get_conv_template_instance(model)
    template.system_message = SYSTEM_MESSAGE
    template.append_message(template.roles[0], "<image>\n" + user_question)
    template.append_message(template.roles[1], answer)
    return template.get_prompt()


def inject_image_tokens(text: str, num_patches: int, num_image_token: int) -> str:
    image_tokens = "<img>" + "<IMG_CONTEXT>" * (num_image_token * num_patches) + "</img>"
    if "<image>" not in text:
        fail("프롬프트 내 <image> 토큰을 찾지 못했습니다.")
    return text.replace("<image>", image_tokens, 1)


# =========================
# 3. Dataset / collator
# =========================
class TrainDataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_root: Path, tokenizer: Any, model: Any):
        self.df = df.reset_index(drop=True)
        self.data_root = data_root
        self.tokenizer = tokenizer
        self.model = model

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        image_path = self.data_root / str(row["path"])
        pixel_values = load_image_tensor(image_path)

        answer = str(row["answer"]).strip().lower()
        if answer not in list("abcd"):
            fail(f"정답이 a/b/c/d가 아닙니다: id={row['id']}, answer={answer}")

        user_question = build_user_question(row)
        prompt_only = build_conversation_text(self.model, user_question, answer=None)
        prompt_with_answer = build_conversation_text(self.model, user_question, answer=answer)

        num_patches = pixel_values.shape[0]
        prompt_only = inject_image_tokens(prompt_only, num_patches, self.model.num_image_token)
        prompt_with_answer = inject_image_tokens(prompt_with_answer, num_patches, self.model.num_image_token)

        enc_prompt = self.tokenizer(prompt_only, return_tensors="pt")
        enc_full = self.tokenizer(prompt_with_answer, return_tensors="pt")

        input_ids = enc_full["input_ids"][0]
        attention_mask = enc_full["attention_mask"][0]
        labels = input_ids.clone()

        prompt_len = enc_prompt["input_ids"].shape[1]
        labels[:prompt_len] = -100

        return {
            "id": str(row["id"]),
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


class EvalDataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_root: Path):
        self.df = df.reset_index(drop=True)
        self.data_root = data_root

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        return {
            "id": str(row["id"]),
            "path": str(self.data_root / str(row["path"])),
            "question": str(row["question"]),
            "a": str(row["a"]),
            "b": str(row["b"]),
            "c": str(row["c"]),
            "d": str(row["d"]),
            "answer": str(row["answer"]).strip().lower() if "answer" in row else None,
        }


@dataclass
class TrainCollator:
    tokenizer: Any

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        pad_id = self.tokenizer.pad_token_id
        if pad_id is None:
            fail("tokenizer.pad_token_id 가 없습니다. pad_token 설정을 먼저 확인하세요.")

        max_len = max(x["input_ids"].shape[0] for x in batch)

        input_ids_list = []
        attention_mask_list = []
        labels_list = []
        pixel_values_list = []

        for sample in batch:
            seq_len = sample["input_ids"].shape[0]
            pad_len = max_len - seq_len

            input_ids = torch.cat([
                sample["input_ids"],
                torch.full((pad_len,), pad_id, dtype=torch.long),
            ], dim=0)
            attention_mask = torch.cat([
                sample["attention_mask"],
                torch.zeros((pad_len,), dtype=torch.long),
            ], dim=0)
            labels = torch.cat([
                sample["labels"],
                torch.full((pad_len,), -100, dtype=torch.long),
            ], dim=0)

            input_ids_list.append(input_ids)
            attention_mask_list.append(attention_mask)
            labels_list.append(labels)
            pixel_values_list.append(sample["pixel_values"])

        pixel_values = torch.cat(pixel_values_list, dim=0)
        image_flags = torch.ones((pixel_values.shape[0], 1), dtype=torch.long)

        return {
            "pixel_values": pixel_values,
            "input_ids": torch.stack(input_ids_list, dim=0),
            "attention_mask": torch.stack(attention_mask_list, dim=0),
            "labels": torch.stack(labels_list, dim=0),
            "image_flags": image_flags,
        }


# =========================
# 4. Model / tokenizer init
# =========================
def get_target_modules_from_arch(model: Any) -> List[str]:
    arch = getattr(model, "llm_arch_name", None)
    if arch is None:
        fail("model.llm_arch_name 을 찾지 못했습니다.")

    if arch == "InternLM2ForCausalLM":
        return [
            "attention.wqkv",
            "attention.wo",
            "feed_forward.w1",
            "feed_forward.w2",
            "feed_forward.w3",
        ]
    if arch == "Phi3ForCausalLM":
        return [
            "mlp.down_proj",
            "mlp.gate_up_proj",
            "self_attn.o_proj",
            "self_attn.qkv_proj",
        ]
    if arch in ["Qwen2ForCausalLM", "LlamaForCausalLM"]:
        return [
            "self_attn.q_proj",
            "self_attn.k_proj",
            "self_attn.v_proj",
            "self_attn.o_proj",
            "mlp.gate_proj",
            "mlp.down_proj",
            "mlp.up_proj",
        ]

    fail(f"지원하지 않는 LLM 아키텍처입니다: {arch}")


def init_model_and_tokenizer(dtype: torch.dtype) -> Tuple[Any, Any]:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=dtype,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        use_fast=False,
    )

    if tokenizer.pad_token is None:
        if tokenizer.eos_token is None:
            fail("tokenizer.pad_token 과 eos_token 이 모두 없습니다.")
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModel.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        torch_dtype=dtype,
        quantization_config=quant_config,
        device_map={"": 0},   # fixed single-GPU placement; do NOT move again with .to(...)
        use_flash_attn=USE_FLASH_ATTN,
    )

    if not hasattr(model, "language_model"):
        fail("InternVL outer model 안에 language_model 이 없습니다.")
    if not hasattr(model, "vision_model"):
        fail("InternVL outer model 안에 vision_model 이 없습니다.")
    if not hasattr(model, "mlp1"):
        fail("InternVL outer model 안에 mlp1 이 없습니다.")
    if not hasattr(model, "num_image_token"):
        fail("InternVL model.num_image_token 이 없습니다.")

    # Freeze vision backbone and projector as in official 2nd finetune shell
    for p in model.vision_model.parameters():
        p.requires_grad = False
    for p in model.mlp1.parameters():
        p.requires_grad = False

    model.language_model = prepare_model_for_kbit_training(
        model.language_model,
        use_gradient_checkpointing=True,
    )

    target_modules = get_target_modules_from_arch(model)
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=target_modules,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model.language_model = get_peft_model(model.language_model, lora_config)
    model.language_model.print_trainable_parameters()

    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    model.train()
    return model, tokenizer


# =========================
# 5. Multiple-choice scoring
# =========================
def get_choice_token_ids(tokenizer: Any) -> Dict[str, int]:
    out = {}
    for ch in "abcd":
        ids = tokenizer.encode(ch, add_special_tokens=False)
        if len(ids) != 1:
            fail(f"선지 '{ch}' 가 단일 토큰이 아닙니다. token ids={ids}")
        out[ch] = ids[0]
    return out


@torch.no_grad()
def score_choices_next_token(
    model: Any,
    tokenizer: Any,
    image_path: str,
    question: str,
    a: str,
    b: str,
    c: str,
    d: str,
    choice_token_ids: Dict[str, int],
    dtype: torch.dtype,
) -> Dict[str, float]:
    pixel_values = load_image_tensor(Path(image_path))
    user_question = (
        f"{question}\n\n" +
        PROMPT_SUFFIX.format(a=a, b=b, c=c, d=d)
    )
    prompt_only = build_conversation_text(model, user_question, answer=None)
    prompt_only = inject_image_tokens(prompt_only, pixel_values.shape[0], model.num_image_token)

    enc = tokenizer(prompt_only, return_tensors="pt")
    input_ids = enc["input_ids"].cuda()
    attention_mask = enc["attention_mask"].cuda()
    pixel_values = pixel_values.to(device="cuda", dtype=dtype)
    image_flags = torch.ones((pixel_values.shape[0], 1), dtype=torch.long, device="cuda")

    outputs = model(
        pixel_values=pixel_values,
        input_ids=input_ids,
        attention_mask=attention_mask,
        image_flags=image_flags,
        return_dict=True,
    )
    next_token_logits = outputs.logits[0, -1]
    scores = {ch: float(next_token_logits[token_id].item()) for ch, token_id in choice_token_ids.items()}
    return scores


@torch.no_grad()
def predict_one(
    model: Any,
    tokenizer: Any,
    sample: Dict[str, Any],
    choice_token_ids: Dict[str, int],
    dtype: torch.dtype,
) -> str:
    scores = score_choices_next_token(
        model=model,
        tokenizer=tokenizer,
        image_path=sample["path"],
        question=sample["question"],
        a=sample["a"],
        b=sample["b"],
        c=sample["c"],
        d=sample["d"],
        choice_token_ids=choice_token_ids,
        dtype=dtype,
    )
    pred = max(scores.items(), key=lambda x: x[1])[0]
    if pred not in list("abcd"):
        fail(f"예측 결과가 a/b/c/d 가 아닙니다: {pred}")
    return pred


@torch.no_grad()
def evaluate_accuracy(
    model: Any,
    tokenizer: Any,
    eval_ds: EvalDataset,
    choice_token_ids: Dict[str, int],
    dtype: torch.dtype,
    desc: str = "valid",
) -> float:
    model.eval()
    correct = 0
    total = 0
    iterator = tqdm(range(len(eval_ds)), desc=desc, leave=False)
    for idx in iterator:
        sample = eval_ds[idx]
        if sample["answer"] is None:
            fail("검증셋에 answer 컬럼이 없습니다.")
        pred = predict_one(model, tokenizer, sample, choice_token_ids, dtype)
        correct += int(pred == sample["answer"])
        total += 1
        iterator.set_postfix(acc=f"{correct / max(total, 1):.4f}")
    model.train()
    return correct / max(total, 1)


# =========================
# 6. Training loop
# =========================
def save_adapter_and_tokenizer(model: Any, tokenizer: Any, save_dir: str) -> None:
    os.makedirs(save_dir, exist_ok=True)
    model.language_model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)

    meta = {
        "base_model": MODEL_ID,
        "image_size": IMAGE_SIZE,
        "max_patches": MAX_PATCHES,
        "use_thumbnail": USE_THUMBNAIL,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "system_message": SYSTEM_MESSAGE,
    }
    with open(os.path.join(save_dir, "run_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)


def train() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    set_seed(SEED)
    dtype = detect_dtype()
    print("Using dtype:", dtype)

    data_root = resolve_data_root()
    print("Data root:", data_root)

    train_df = pd.read_csv(data_root / TRAIN_CSV_NAME)
    test_df = pd.read_csv(data_root / TEST_CSV_NAME)
    validate_dataframe(train_df, is_train=True)
    validate_dataframe(test_df, is_train=False)

    if MAX_SAMPLES is not None:
        if MAX_SAMPLES <= 0:
            fail("MAX_SAMPLES 는 양의 정수여야 합니다.")
        train_df = train_df.sample(min(MAX_SAMPLES, len(train_df)), random_state=SEED).reset_index(drop=True)

    train_part, valid_part = train_test_split(
        train_df,
        test_size=VAL_SIZE,
        random_state=SEED,
        shuffle=True,
        stratify=train_df["answer"],
    )
    train_part = train_part.reset_index(drop=True)
    valid_part = valid_part.reset_index(drop=True)

    model, tokenizer = init_model_and_tokenizer(dtype)
    choice_token_ids = get_choice_token_ids(tokenizer)

    train_ds = TrainDataset(train_part, data_root, tokenizer, model)
    valid_ds = EvalDataset(valid_part, data_root)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        collate_fn=TrainCollator(tokenizer),
    )

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if len(trainable_params) == 0:
        fail("학습 가능한 파라미터가 0개입니다. LoRA 적용에 실패했습니다.")

    optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)

    total_optimizer_steps = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * NUM_EPOCHS
    warmup_steps = max(1, int(total_optimizer_steps * WARMUP_RATIO))
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_optimizer_steps,
    )

    best_val_acc = -1.0
    best_dir = os.path.join(OUTPUT_DIR, "best_adapter")
    log_rows = []

    global_step = 0
    optimizer.zero_grad(set_to_none=True)

    for epoch in range(NUM_EPOCHS):
        model.train()
        progress = tqdm(train_loader, desc=f"train epoch {epoch + 1}", leave=True)
        running_loss = 0.0
        seen_batches = 0

        for step, batch in enumerate(progress, start=1):
            input_ids = batch["input_ids"].cuda(non_blocking=True)
            attention_mask = batch["attention_mask"].cuda(non_blocking=True)
            labels = batch["labels"].cuda(non_blocking=True)
            image_flags = batch["image_flags"].cuda(non_blocking=True)
            pixel_values = batch["pixel_values"].to(device="cuda", dtype=dtype, non_blocking=True)

            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                image_flags=image_flags,
                labels=labels,
                return_dict=True,
            )
            loss = outputs.loss

            if torch.isnan(loss).any() or torch.isinf(loss).any():
                fail(f"loss 가 NaN/Inf 입니다. global_step={global_step}, epoch={epoch + 1}, step={step}")

            loss_for_backward = loss / GRAD_ACCUM_STEPS
            loss_for_backward.backward()

            running_loss += float(loss.item())
            seen_batches += 1

            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
                torch.nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD_NORM)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

            avg_loss = running_loss / max(seen_batches, 1)
            progress.set_postfix(loss=f"{avg_loss:.4f}", gstep=global_step)

        val_acc = evaluate_accuracy(
            model=model,
            tokenizer=tokenizer,
            eval_ds=valid_ds,
            choice_token_ids=choice_token_ids,
            dtype=dtype,
            desc=f"valid epoch {epoch + 1}",
        )

        epoch_loss = running_loss / max(seen_batches, 1)
        print(f"[Epoch {epoch + 1}] train_loss={epoch_loss:.6f} val_acc={val_acc:.6f}")

        log_rows.append({
            "epoch": epoch + 1,
            "train_loss": epoch_loss,
            "val_acc": val_acc,
            "global_step": global_step,
        })
        pd.DataFrame(log_rows).to_csv(os.path.join(OUTPUT_DIR, "train_log.csv"), index=False)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            save_adapter_and_tokenizer(model, tokenizer, best_dir)
            print(f"Best adapter updated: {best_dir} (val_acc={best_val_acc:.6f})")

        gc.collect()
        torch.cuda.empty_cache()

    print("Training done. Best val_acc =", best_val_acc)
    run_inference(
        data_root=data_root,
        model=model,
        tokenizer=tokenizer,
        test_df=test_df,
        choice_token_ids=choice_token_ids,
        dtype=dtype,
    )


# =========================
# 7. Inference / submission
# =========================
@torch.no_grad()
def run_inference(
    data_root: Path,
    model: Any,
    tokenizer: Any,
    test_df: pd.DataFrame,
    choice_token_ids: Dict[str, int],
    dtype: torch.dtype,
) -> None:
    validate_dataframe(test_df, is_train=False)
    model.eval()

    preds = []
    iterator = tqdm(range(len(test_df)), desc="test inference", leave=True)
    for idx in iterator:
        row = test_df.iloc[idx]
        sample = {
            "path": str(data_root / str(row["path"])),
            "question": str(row["question"]),
            "a": str(row["a"]),
            "b": str(row["b"]),
            "c": str(row["c"]),
            "d": str(row["d"]),
        }
        pred = predict_one(model, tokenizer, sample, choice_token_ids, dtype)
        preds.append(pred)

    submission = pd.DataFrame({
        "id": test_df["id"].astype(str),
        "answer": preds,
    })
    out_path = os.path.join(OUTPUT_DIR, SAVE_SUBMISSION_NAME)
    submission.to_csv(out_path, index=False)
    print("Saved submission:", out_path)
    print(submission.head())


if __name__ == "__main__":
    train()
